# Install Libraries

In [ ]:
!pip install -q torch==2.4.1 torchvision==0.19.1 transformers==4.44.2 pillow==10.4.0
!pip install pdfplumber
!pip install reportlab
!pip install langgraph langchain langchain-core

# Import Libraries

In [1]:
from transformers import pipeline
import pdfplumber
import re
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from langgraph.graph import StateGraph, END
from typing import TypedDict

In [30]:
!ls

drive  sample_data


In [31]:
%cd /content/drive/MyDrive/Colab Notebooks/dataset

/content/drive/MyDrive/Colab Notebooks/dataset


# PDF Extractor

In [3]:
def extract_text_from_pdf(pdf_path):
    """
    Extracts text from every page of a PDF.
    """

    full_text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            text = page.extract_text()

            if text:
                full_text += text + "\n"

    return full_text

# Text Cleaner

In [4]:
def clean_text(text):
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # Remove special characters (keep basic punctuation)
    text = re.sub(r'[^\w\s.,;:!?()-]', '', text)

    # Strip leading/trailing spaces
    text = text.strip()

    return text

# Chunking

In [5]:
def chunk_text(text, max_words=300):
    words = text.split()
    return [
        " ".join(words[i:i + max_words])
        for i in range(0, len(words), max_words)
    ]

# Load Models

In [6]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)
draft_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

# Summarizer Agent

In [7]:
def summarize_document(chunks):
    summaries = []

    for chunk in chunks:

        input_length = len(chunk.split())

        max_len = min(120, input_length - 5)
        min_len = min(30, max_len - 5)

        summary = summarizer(
            chunk,
            max_length=max_len,
            min_length=max(10, min_len),
            do_sample=False
        )[0]["summary_text"]

        summaries.append(summary)

    return " ".join(summaries)

# Classificatin Agent

In [8]:
def classify_document(text):

    labels = [
        "legal contract agreement",
        "property lease agreement",
        "employment contract",
        "non disclosure agreement",
        "memorandum of understanding",
        "service agreement",
        "power of attorney",
        "affidavit",
        "will and testament",
        "court judgement",
        "legal notice",
        "police complaint",
        "partnership agreement"
    ]

    result = classifier(
        text,
        labels,
        hypothesis_template="This document is a {}."
    )

    return {
        "document_type": result["labels"][0],
        "confidence": result["scores"][0]
    }

# Analyzer Agent


In [9]:
def analyze_risks(text):

    text_lower = text.lower()

    risk_patterns = {
        "HIGH": [
            r"all losses",
            r"without limitation",
            r"unlimited liability",
            r"indemnif\w+",
            r"held harmless",
            r"held liable",
        ],
        "MEDIUM": [
            r"terminate anytime",
            r"without notice",
            r"no notice",
            r"immediately",
            r"penalty",
            r"fine",
            r"attorney",
            r"no refund"
        ]
    }

    risks = []

    for level, patterns in risk_patterns.items():

        for pattern in patterns:

            matches = re.finditer(pattern, text_lower)

            for match in matches:

                start = max(0, match.start()-100)
                end = min(len(text), match.end()+100)

                clause = text[start:end]

                risks.append({
                    "risk": level,
                    "pattern": pattern,
                    "clause": clause
                })

    return risks

# Drafting agent

In [10]:
def drafting_agent(clause, risk_level):

    prompt = f"""
You are an expert legal assistant.

Rewrite ONLY the clause below.

Requirements:
- Keep the same meaning.
- Reduce legal risk.
- Use professional legal language.
- Return only the revised clause.
- Do NOT explain anything.
- Do NOT number the answer.

Risk Level:
{risk_level}

Clause:
{clause}

Revised Clause:
"""

    result = draft_model(
        prompt,
        max_length=150,
        do_sample=False
    )

    return result[0]["generated_text"].strip()

# Report generator

In [11]:
def generate_report(document_type, confidence, summary, risks):

    report = "\n========== LEGAL DOCUMENT ANALYSIS REPORT ==========\n\n"

    report += f"Document Type : {document_type}\n"
    report += f"Confidence    : {confidence:.2f}\n\n"

    report += "--------------- SUMMARY ---------------\n"
    report += summary + "\n\n"

    report += "------------- RISK ANALYSIS ------------\n"

    if risks:
        for risk in risks:


            report += f"Risk Level : {risk['risk']}\n"

            report += f"Detected Clause:\n{risk['clause']}\n\n"

            safer_clause = drafting_agent(
                risk["clause"],
                risk["risk"]
            )

            report += "Suggested Revision:\n"

            report += safer_clause

            report += "\n"

            report += "-"*60 + "\n"

    else:
        report += "No risks detected.\n"

    report += "\n===============================================\n"

    return report

# Save report as txt

In [12]:
def save_report_txt(report, filename="reports/legal_analysis_report.txtx"):

    with open(filename, "w", encoding="utf-8") as file:
        file.write(report)

    print(f"Report saved as: {filename}")

# Orchestration (main pipeline)

In [16]:
def legal_document_analysis(pdf_path):

    # Step 1: Extract Text
    raw_text = extract_text_from_pdf(pdf_path)

    # Step 2: Clean Text
    cleaned_text = clean_text(raw_text)

    # Step 3: Chunk Text
    chunks = chunk_text(cleaned_text)

    # Step 4: Summarize
    summary = summarize_document(chunks)

    # Step 5: Classify
    document_info = classify_document(cleaned_text)

    # Step 6: Risk Analysis
    risks = analyze_risks(cleaned_text)

    # Step 7: Generate Report
    report = generate_report(
        document_info["document_type"],
        document_info["confidence"],
        summary,
        risks
    )

    save_report_txt(report)

    return report

# States

In [17]:
class AgentState(TypedDict):

    pdf_path: str

    raw_text: str

    cleaned_text: str

    chunks: list

    summary: str

    document_type: str

    confidence: float

    risks: list

    report: str

# Agent 1 Extracter

In [18]:
def extraction_node(state):

    raw_text = extract_text_from_pdf(state["pdf_path"])

    return {
        "raw_text": raw_text
    }

# Agent 2 Cleaner

In [19]:
def cleaning_node(state):

    cleaned_text = clean_text(
        state["raw_text"]
    )

    chunks = chunk_text(
        cleaned_text
    )

    return {
        "cleaned_text": cleaned_text,
        "chunks": chunks
    }

# Agent 3 Summarizer

In [20]:
def summarizer_node(state):

    summary = summarize_document(
        state["chunks"]
    )

    return {
        "summary": summary
    }

# Agent 4 Classifier

In [21]:
def classifier_node(state):

    result = classify_document(
        state["cleaned_text"]
    )

    return {
        "document_type": result["document_type"],
        "confidence": result["confidence"]
    }

# Agent 5 Risk Analyzer

In [22]:
def risk_node(state):

    risks = analyze_risks(
        state["cleaned_text"]
    )

    return {
        "risks": risks
    }

# Agent 6 Report Generator

In [23]:
def report_node(state):

    report = generate_report(
        state["document_type"],
        state["confidence"],
        state["summary"],
        state["risks"]
    )

    return {
        "report": report
    }

# Build Graph

In [24]:
graph = StateGraph(AgentState)

graph.add_node(
    "extractor",
    extraction_node
)

graph.add_node(
    "cleaner",
    cleaning_node
)

graph.add_node(
    "summarizer",
    summarizer_node
)

graph.add_node(
    "classifier",
    classifier_node
)

graph.add_node(
    "risk",
    risk_node
)

graph.add_node(
    "report",
    report_node
)

# Connect Agents

In [25]:
graph.set_entry_point(
    "extractor"
)

graph.add_edge(
    "extractor",
    "cleaner"
)

graph.add_edge(
    "cleaner",
    "summarizer"
)

graph.add_edge(
    "summarizer",
    "classifier"
)

graph.add_edge(
    "classifier",
    "risk"
)

graph.add_edge(
    "risk",
    "report"
)

graph.add_edge(
    "report",
    END
)

# Compile

In [26]:
app = graph.compile()

# Run

In [32]:
result = app.invoke({

    "pdf_path":
    "Rental Agreement Template.pdf"

})

print(
    result["report"]
)


========== LEGAL DOCUMENT ANALYSIS REPORT ==========

Document Type : partnership agreement
Confidence    : 0.13

--------------- SUMMARY ---------------
This is an agreement for a Short-Term Lease according to the terms set out below. The Landlord, the Tenant and the Premises are as set out in Schedule 1 of this Agreement. The Tenants shall maintain the premises in a good, clean, and ready to rent condition, and use the premises only in a careful and lawful manner. The Tenants will not sublet the Premises without the express written permission of the Landlord. The Tenants shall not create noise or disturbances likely to disturb or annoy the surrounding property owners. Creating a disturbance of the above nature shall be grounds for immediate termination of this agreement. The Tenants will not smoke inside the property without the express written permission of the Landlord. There shall be no refunds of rents due to shortened stays or ruined expectations because of weather conditions. 